# Experiment 6: Triple Extraction from Text (Wikipedia)

**Goal:** Extract RDF triples from Wikipedia article text using NER and LLM prompting — the two techniques taught in Week 7.

**Key questions:**
- What entities does SpaCy NER extract from a music history article?
- Can we use LLM-style prompting to extract structured triples?
- What relation types can we extract from text that aren't in structured data?
- How do we link extracted entities back to MusicBrainz MBIDs?

## Setup

In [1]:
import requests
import json

WIKI_API = "https://en.wikipedia.org/w/api.php"
HEADERS = {"User-Agent": "KE-CW2-MusicHistory/0.1 (lucas@example.com)"}

def wiki_get(params):
    params["format"] = "json"
    resp = requests.get(WIKI_API, params=params, headers=HEADERS)
    resp.raise_for_status()
    return resp.json()

def pp(data):
    print(json.dumps(data, indent=2, default=str))

## 1. Fetch Article Text

Get the full David Bowie article and a targeted section (Music career intro).

In [2]:
# Fetch the intro (first few paragraphs)
result = wiki_get({
    "action": "query",
    "titles": "David Bowie",
    "prop": "extracts",
    "explaintext": True,
    "exintro": True,
    "exlimit": 1
})

page = list(result["query"]["pages"].values())[0]
intro_text = page.get("extract", "")

print(f"Intro length: {len(intro_text)} chars")
print(f"\n{intro_text[:2000]}")

Intro length: 3051 chars

David Robert Jones (8 January 1947 – 10 January 2016), known as David Bowie, was an English singer, songwriter and actor. Regarded as among the most influential musicians of the 20th century, Bowie received particular acclaim for his work in the 1970s. His career was marked by reinvention and visual presentation, and his music and stagecraft have had a significant impact on popular music.
Bowie studied art, music and design before embarking on a music career in 1962. He released a string of unsuccessful singles with local bands and a self-titled solo album (1967) before achieving his first top-five entry on the UK singles chart with "Space Oddity" (1969). After a period of experimentation, he re-emerged in 1972 during the glam rock era with the alter ego Ziggy Stardust. The single "Starman" and its album The Rise and Fall of Ziggy Stardust and the Spiders from Mars (1972) won him widespread popularity. In 1975, Bowie's style shifted towards a sound he characte

## 2. SpaCy NER — Entity Extraction

Week 7 technique: use SpaCy to extract named entities from text. Let's see what it finds.

In [3]:
# Install spacy and download model if needed
# !pip install spacy
# !python -m spacy download en_core_web_sm

import spacy

nlp = spacy.load("en_core_web_sm")
print(f"SpaCy model loaded: {nlp.meta['name']}")

SpaCy model loaded: core_web_sm


In [4]:
# Run NER on the intro text
doc = nlp(intro_text)

print(f"Entities found: {len(doc.ents)}\n")

# Group by entity type
from collections import defaultdict
entities_by_type = defaultdict(list)

for ent in doc.ents:
    entities_by_type[ent.label_].append(ent.text)

for label, ents in sorted(entities_by_type.items()):
    # Deduplicate while preserving order
    unique = list(dict.fromkeys(ents))
    print(f"\n{label} ({len(unique)} unique):")
    for e in unique[:15]:
        print(f"  - {e}")
    if len(unique) > 15:
        print(f"  ... and {len(unique) - 15} more")

Entities found: 88


CARDINAL (6 unique):
  - five
  - one
  - three
  - 200
  - six
  - four

DATE (31 unique):
  - 8 January 1947
  - January 2016
  - the 20th century
  - the 1970s
  - 1962
  - 1967
  - 1969
  - 1972
  - 1975
  - 1976
  - 1977
  - 1979
  - the late 1970s
  - 1980
  - 1981
  ... and 16 more

GPE (2 unique):
  - UK
  - US

LOC (1 unique):
  - Mars

NORP (1 unique):
  - English

ORDINAL (1 unique):
  - first

ORG (7 unique):
  - the Berlin Trilogy
  - Scary Monsters
  - Super Creeps
  - Basquiat
  - Brit Awards
  - the Rock and Roll Hall of Fame
  - Rolling Stone

PERSON (12 unique):
  - David Robert Jones
  - David Bowie
  - Bowie
  - Ziggy Stardust
  - Young Americans
  - Low
  - Brian Eno
  - Tin Machine
  - Lawrence
  - Labyrinth
  - Twin Peaks
  - Blackstar

PRODUCT (1 unique):
  - Starman

WORK_OF_ART (6 unique):
  - Space Oddity
  - Fame
  - The Man Who Fell to Earth
  - Ashes to Ashes
  - Under Pressure
  - Merry Christmas


### 2a. Analyse NER Quality

Which entities are useful for our KG? Which are noise?

In [5]:
# Map SpaCy entity types to our ontology
ONTOLOGY_MAP = {
    "PERSON": "Artist (potential)",
    "ORG": "Band / Record Label / Organisation",
    "GPE": "Country / Place",
    "LOC": "Venue / Place",
    "DATE": "Era / releaseDate / activeYears",
    "WORK_OF_ART": "Album / Track title",
    "EVENT": "Performance / Award ceremony",
    "CARDINAL": "Numeric (mostly noise)",
    "ORDINAL": "Ranking (mostly noise)",
    "MONEY": "Not in ontology",
    "NORP": "Nationality/group (e.g., English, British)",
}

print("SpaCy Entity Type → Ontology Mapping:\n")
for label in sorted(entities_by_type.keys()):
    mapping = ONTOLOGY_MAP.get(label, "Unknown")
    count = len(set(entities_by_type[label]))
    print(f"  {label:15s} → {mapping:40s} ({count} unique)")

SpaCy Entity Type → Ontology Mapping:

  CARDINAL        → Numeric (mostly noise)                   (6 unique)
  DATE            → Era / releaseDate / activeYears          (31 unique)
  GPE             → Country / Place                          (2 unique)
  LOC             → Venue / Place                            (1 unique)
  NORP            → Nationality/group (e.g., English, British) (1 unique)
  ORDINAL         → Ranking (mostly noise)                   (1 unique)
  ORG             → Band / Record Label / Organisation       (7 unique)
  PERSON          → Artist (potential)                       (12 unique)
  PRODUCT         → Unknown                                  (1 unique)
  WORK_OF_ART     → Album / Track title                      (6 unique)


### 2b. Extract Relevant Entities Only

Filter to just the entity types that map to our ontology.

In [6]:
USEFUL_TYPES = {"PERSON", "ORG", "GPE", "LOC", "DATE", "WORK_OF_ART", "EVENT"}

relevant_entities = []
for ent in doc.ents:
    if ent.label_ in USEFUL_TYPES:
        relevant_entities.append({
            "text": ent.text,
            "type": ent.label_,
            "start": ent.start_char,
            "end": ent.end_char
        })

print(f"Relevant entities: {len(relevant_entities)} (from {len(doc.ents)} total)\n")

# Show unique relevant entities
seen = set()
for e in relevant_entities:
    key = (e['text'], e['type'])
    if key not in seen:
        seen.add(key)
        print(f"  [{e['type']:15s}] {e['text']}")

Relevant entities: 75 (from 88 total)

  [PERSON         ] David Robert Jones
  [DATE           ] 8 January 1947
  [DATE           ] January 2016
  [PERSON         ] David Bowie
  [DATE           ] the 20th century
  [PERSON         ] Bowie
  [DATE           ] the 1970s
  [DATE           ] 1962
  [DATE           ] 1967
  [GPE            ] UK
  [WORK_OF_ART    ] Space Oddity
  [DATE           ] 1969
  [DATE           ] 1972
  [PERSON         ] Ziggy Stardust
  [LOC            ] Mars
  [DATE           ] 1975
  [GPE            ] US
  [WORK_OF_ART    ] Fame
  [PERSON         ] Young Americans
  [DATE           ] 1976
  [WORK_OF_ART    ] The Man Who Fell to Earth
  [DATE           ] 1977
  [PERSON         ] Low
  [PERSON         ] Brian Eno
  [ORG            ] the Berlin Trilogy
  [DATE           ] 1979
  [DATE           ] the late 1970s
  [DATE           ] 1980
  [WORK_OF_ART    ] Ashes to Ashes
  [ORG            ] Scary Monsters
  [ORG            ] Super Creeps
  [WORK_OF_ART    ] Under P

## 3. LLM-Style Structured Extraction

Week 7 technique: use prompting to extract structured triples from text.

Since we can't call an LLM directly from this notebook, we'll **design the prompt** and **simulate the expected output format**. You can then run this prompt against ChatGPT/Claude.

In [7]:
# Take a manageable chunk of text for prompting
# Use the first 1500 chars of the intro
text_chunk = intro_text[:1500]

prompt = f"""You are a knowledge engineer extracting structured information from music history text.

Given the following text about a musician, extract all factual relationships as triples in the format:
(Subject, Predicate, Object)

Use these predicate types:
- hasGenre: artist/album belongs to a genre
- released: artist released an album
- releaseDate: album was released on a date
- memberOf: artist is member of a band
- collaboratedWith: artist collaborated with another artist
- influencedBy: artist was influenced by another artist
- bornIn: artist was born in a place
- diedIn: artist died in a place
- birthDate: artist's date of birth
- deathDate: artist's date of death
- wonAward: artist/album won an award
- signedTo: artist signed to a record label
- performedAt: artist performed at a venue/event
- countryOfOrigin: artist is from a country
- realName: artist's birth/real name
- playsInstrument: artist plays an instrument

TEXT:
\"\"\" 
{text_chunk}
\"\"\"

Extract all triples. Output as JSON array of objects with keys: subject, predicate, object.
"""

print("=" * 60)
print("PROMPT FOR LLM TRIPLE EXTRACTION")
print("=" * 60)
print(prompt)

PROMPT FOR LLM TRIPLE EXTRACTION
You are a knowledge engineer extracting structured information from music history text.

Given the following text about a musician, extract all factual relationships as triples in the format:
(Subject, Predicate, Object)

Use these predicate types:
- hasGenre: artist/album belongs to a genre
- released: artist released an album
- releaseDate: album was released on a date
- memberOf: artist is member of a band
- collaboratedWith: artist collaborated with another artist
- influencedBy: artist was influenced by another artist
- bornIn: artist was born in a place
- diedIn: artist died in a place
- birthDate: artist's date of birth
- deathDate: artist's date of death
- wonAward: artist/album won an award
- signedTo: artist signed to a record label
- performedAt: artist performed at a venue/event
- countryOfOrigin: artist is from a country
- realName: artist's birth/real name
- playsInstrument: artist plays an instrument

TEXT:
""" 
David Robert Jones (8 Janu

In [8]:
# Expected output format (manually created based on reading the text)
# This is what we'd expect an LLM to return

expected_triples = [
    {"subject": "David Bowie", "predicate": "realName", "object": "David Robert Jones"},
    {"subject": "David Bowie", "predicate": "birthDate", "object": "1947-01-08"},
    {"subject": "David Bowie", "predicate": "deathDate", "object": "2016-01-10"},
    {"subject": "David Bowie", "predicate": "countryOfOrigin", "object": "England"},
    {"subject": "David Bowie", "predicate": "hasGenre", "object": "glam rock"},
    {"subject": "David Bowie", "predicate": "hasGenre", "object": "plastic soul"},
    {"subject": "David Bowie", "predicate": "released", "object": "Space Oddity"},
    {"subject": "David Bowie", "predicate": "released", "object": "The Rise and Fall of Ziggy Stardust and the Spiders from Mars"},
    {"subject": "David Bowie", "predicate": "released", "object": "Young Americans"},
    {"subject": "David Bowie", "predicate": "released", "object": "Station to Station"},
    {"subject": "David Bowie", "predicate": "released", "object": "Low"},
    {"subject": "David Bowie", "predicate": "collaboratedWith", "object": "Brian Eno"},
    {"subject": "David Bowie", "predicate": "collaboratedWith", "object": "Queen"},
    {"subject": "David Bowie", "predicate": "released", "object": "Let's Dance"},
    {"subject": "David Bowie", "predicate": "memberOf", "object": "Tin Machine"},
    {"subject": "David Bowie", "predicate": "released", "object": "Blackstar"},
    {"subject": "David Bowie", "predicate": "released", "object": "The Next Day"},
    {"subject": "David Bowie", "predicate": "wonAward", "object": "Rock and Roll Hall of Fame"},
    {"subject": "Young Americans", "predicate": "releaseDate", "object": "1975"},
    {"subject": "The Rise and Fall of Ziggy Stardust and the Spiders from Mars", "predicate": "releaseDate", "object": "1972"},
]

print(f"Expected triples from text: {len(expected_triples)}\n")
for t in expected_triples:
    print(f"  ({t['subject']}, {t['predicate']}, {t['object']})")

Expected triples from text: 20

  (David Bowie, realName, David Robert Jones)
  (David Bowie, birthDate, 1947-01-08)
  (David Bowie, deathDate, 2016-01-10)
  (David Bowie, countryOfOrigin, England)
  (David Bowie, hasGenre, glam rock)
  (David Bowie, hasGenre, plastic soul)
  (David Bowie, released, Space Oddity)
  (David Bowie, released, The Rise and Fall of Ziggy Stardust and the Spiders from Mars)
  (David Bowie, released, Young Americans)
  (David Bowie, released, Station to Station)
  (David Bowie, released, Low)
  (David Bowie, collaboratedWith, Brian Eno)
  (David Bowie, collaboratedWith, Queen)
  (David Bowie, released, Let's Dance)
  (David Bowie, memberOf, Tin Machine)
  (David Bowie, released, Blackstar)
  (David Bowie, released, The Next Day)
  (David Bowie, wonAward, Rock and Roll Hall of Fame)
  (Young Americans, releaseDate, 1975)
  (The Rise and Fall of Ziggy Stardust and the Spiders from Mars, releaseDate, 1972)


## 4. Compare: SpaCy NER vs LLM Extraction

What can each method extract?

In [9]:
# What SpaCy found (entities only, no relations)
spacy_persons = set(e['text'] for e in relevant_entities if e['type'] == 'PERSON')
spacy_orgs = set(e['text'] for e in relevant_entities if e['type'] == 'ORG')
spacy_works = set(e['text'] for e in relevant_entities if e['type'] == 'WORK_OF_ART')
spacy_dates = set(e['text'] for e in relevant_entities if e['type'] == 'DATE')
spacy_places = set(e['text'] for e in relevant_entities if e['type'] in ('GPE', 'LOC'))

# What LLM extraction would find (entities + relations)
llm_subjects = set(t['subject'] for t in expected_triples)
llm_objects = set(t['object'] for t in expected_triples)
llm_predicates = set(t['predicate'] for t in expected_triples)

print("COMPARISON: SpaCy NER vs LLM Extraction\n")
print(f"{'':30s} {'SpaCy NER':>15s} {'LLM Extraction':>15s}")
print("-" * 65)
print(f"{'People/Artists':30s} {len(spacy_persons):>15d} {len([t for t in expected_triples if t['predicate'] in ('collaboratedWith', 'memberOf')]):>15d}")
print(f"{'Organisations/Bands':30s} {len(spacy_orgs):>15d} {'—':>15s}")
print(f"{'Works (Albums/Tracks)':30s} {len(spacy_works):>15d} {len([t for t in expected_triples if t['predicate'] == 'released']):>15d}")
print(f"{'Dates':30s} {len(spacy_dates):>15d} {len([t for t in expected_triples if 'Date' in t['predicate']]):>15d}")
print(f"{'Places':30s} {len(spacy_places):>15d} {len([t for t in expected_triples if t['predicate'] in ('countryOfOrigin', 'bornIn')]):>15d}")
print(f"{'Relations extracted':30s} {'0 (NER only)':>15s} {len(expected_triples):>15d}")

print("\n" + "=" * 65)
print("KEY INSIGHT: SpaCy finds entities but NOT relations.")
print("LLM extraction finds both entities AND relations (triples).")
print("Best approach: use SpaCy for entity discovery, LLM for relation extraction.")

COMPARISON: SpaCy NER vs LLM Extraction

                                     SpaCy NER  LLM Extraction
-----------------------------------------------------------------
People/Artists                              12               3
Organisations/Bands                          7               —
Works (Albums/Tracks)                        6               8
Dates                                       31               4
Places                                       3               1
Relations extracted               0 (NER only)              20

KEY INSIGHT: SpaCy finds entities but NOT relations.
LLM extraction finds both entities AND relations (triples).
Best approach: use SpaCy for entity discovery, LLM for relation extraction.


## 5. What Text Extraction Adds Over Structured Data

Compare what we got from MusicBrainz/Discogs (Experiment 5) vs what text gives us.

In [10]:
print("TRIPLES ONLY AVAILABLE FROM TEXT (not in MusicBrainz/Discogs):\n")

text_only_triples = [
    ("David Bowie", "collaboratedWith", "Brian Eno", "MB has this but not always — text confirms it"),
    ("David Bowie", "collaboratedWith", "Queen", "MB may have this — text makes it explicit"),
    ("David Bowie", "hasGenre", "plastic soul", "Not a standard genre tag in MB/Discogs"),
    ("David Bowie", "hasGenre", "industrial", "Mentioned in text, may not be in tags"),
    ("David Bowie", "hasGenre", "jungle", "Mentioned in text, may not be in tags"),
    ("Low", "partOf", "Berlin Trilogy", "Album grouping not in structured data"),
    ("Heroes", "partOf", "Berlin Trilogy", "Album grouping not in structured data"),
    ("Lodger", "partOf", "Berlin Trilogy", "Album grouping not in structured data"),
    ("David Bowie", "wonAward", "Grammy Award", "Award data — 6 Grammys mentioned"),
    ("David Bowie", "wonAward", "Brit Award", "Award data — 4 Brits mentioned"),
    ("David Bowie", "wonAward", "Rock and Roll Hall of Fame", "Inducted 1996"),
    ("Blackstar", "chartPosition", "#1 US Billboard 200", "Chart data not in MB/Discogs"),
    ("David Bowie", "alterEgo", "Ziggy Stardust", "Persona — unique to text"),
    ("David Bowie", "alterEgo", "Thin White Duke", "Persona — unique to text"),
]

for s, p, o, note in text_only_triples:
    print(f"  ({s}, {p}, {o})")
    print(f"    → {note}\n")

TRIPLES ONLY AVAILABLE FROM TEXT (not in MusicBrainz/Discogs):

  (David Bowie, collaboratedWith, Brian Eno)
    → MB has this but not always — text confirms it

  (David Bowie, collaboratedWith, Queen)
    → MB may have this — text makes it explicit

  (David Bowie, hasGenre, plastic soul)
    → Not a standard genre tag in MB/Discogs

  (David Bowie, hasGenre, industrial)
    → Mentioned in text, may not be in tags

  (David Bowie, hasGenre, jungle)
    → Mentioned in text, may not be in tags

  (Low, partOf, Berlin Trilogy)
    → Album grouping not in structured data

  (Heroes, partOf, Berlin Trilogy)
    → Album grouping not in structured data

  (Lodger, partOf, Berlin Trilogy)
    → Album grouping not in structured data

  (David Bowie, wonAward, Grammy Award)
    → Award data — 6 Grammys mentioned

  (David Bowie, wonAward, Brit Award)
    → Award data — 4 Brits mentioned

  (David Bowie, wonAward, Rock and Roll Hall of Fame)
    → Inducted 1996

  (Blackstar, chartPosition, #1 

## 6. Entity Linking Challenge

How do we link entities extracted from text back to our KG URIs?

In [11]:
import musicbrainzngs
import time

musicbrainzngs.set_useragent("KE-CW2-MusicHistory", "0.1", "lucas@example.com")

# Try to link text-extracted entities to MusicBrainz
entities_to_link = ["Brian Eno", "Queen", "Tin Machine", "Ziggy Stardust"]

print("Entity Linking: Text → MusicBrainz\n")

for entity_name in entities_to_link:
    time.sleep(1.1)
    try:
        search = musicbrainzngs.search_artists(artist=entity_name, limit=1)
        if search["artist-list"]:
            match = search["artist-list"][0]
            score = match.get("ext:score", "?")
            print(f"  '{entity_name}' → {match['name']} (MBID: {match['id']}, score: {score})")
        else:
            print(f"  '{entity_name}' → NOT FOUND")
    except Exception as e:
        print(f"  '{entity_name}' → ERROR: {e}")

Entity Linking: Text → MusicBrainz

  'Brian Eno' → Brian Eno (MBID: ff95eb47-41c4-4f7f-a104-cdc30f02e872, score: 100)
  'Queen' → Queen (MBID: 0383dadf-2a4e-4d10-a46a-e9e041da8eb3, score: 100)
  'Tin Machine' → Tin Machine (MBID: 39dfc059-93b1-41a1-af27-3a753c0711d3, score: 100)
  'Ziggy Stardust' → Stardust Revue (MBID: 2bcb6d35-0a15-400e-97e2-832900f21820, score: 100)


## 7. Summary

### SpaCy NER

| Aspect | Observation |
|---|---|
| Entities found | Fill in |
| Useful types | PERSON, ORG, GPE, WORK_OF_ART, DATE |
| Noise types | CARDINAL, ORDINAL, MONEY |
| Relations? | No — NER only finds entities, not relationships |
| Best use | Entity discovery + validation |

### LLM Extraction

| Aspect | Observation |
|---|---|
| Triples extracted | ~20 from intro alone |
| Relations found | collaboratedWith, released, hasGenre, wonAward, memberOf, etc. |
| Unique value | Finds info NOT in structured sources (awards, personas, album groupings) |
| Best use | Full triple extraction from targeted sections |

### Recommended Text Pipeline

```
1. Fetch Wikipedia article via cross-reference (Wikidata → Wikipedia)
2. Extract key sections (Music career, Legacy, Musicianship)
3. Run SpaCy NER for entity discovery
4. Run LLM prompt for triple extraction
5. Link extracted entities to MusicBrainz MBIDs via search
6. Add new triples to the RDF graph
```

## Next Steps

- Integrate text extraction into the main pipeline
- Scale up: process multiple artists
- Build the formal ontology (.ttl) with proper class/property declarations
- Write the 20 SPARQL queries for competency questions